# Phase 2.3 — Product Dataset

Build a canonical product table from Retailrocket product metadata and interaction coverage.

The dataset uses a generic `property` / `value` representation. Product attributes such as price, brand, description, and features are included only when supported by the raw data.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "raw").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH = PROCESSED_DIR / "product_dataset.csv"

EVENTS_PATH = RAW_DIR / "events.csv"
PROPERTIES_1 = RAW_DIR / "item_properties_part1.csv"
PROPERTIES_2 = RAW_DIR / "item_properties_part2.csv"

print("Raw:", RAW_DIR)
print("Output:", OUTPUT_PATH)

Raw: f:\annuspeaks.com\recommendation-system\data\raw
Output: f:\annuspeaks.com\recommendation-system\data\processed\product_dataset.csv


In [2]:
# Build the canonical product catalog from all observed item IDs.

event_items = set()

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=["itemid"],
    chunksize=250_000
):
    event_items.update(chunk["itemid"].dropna().unique())

property_items = set()

for path in [PROPERTIES_1, PROPERTIES_2]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid"],
        chunksize=250_000
    ):
        property_items.update(chunk["itemid"].dropna().unique())

all_items = sorted(event_items | property_items)

products = pd.DataFrame({
    "item_id": all_items
})

products["has_interactions"] = products["item_id"].isin(event_items)
products["has_metadata"] = products["item_id"].isin(property_items)

print("Canonical products:", f"{len(products):,}")
display(products.head())

Canonical products: 466,868


,item_id,has_interactions,has_metadata
0,0,False,True
1,1,False,True
2,2,False,True
3,3,True,True
4,4,True,True


In [3]:
# Collect product properties.

property_parts = []

for path in [PROPERTIES_1, PROPERTIES_2]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid", "property", "value"],
        chunksize=250_000
    ):
        property_parts.append(chunk)

properties = pd.concat(
    property_parts,
    ignore_index=True
)

properties = properties.drop_duplicates(
    subset=["itemid", "property", "value"]
)

print("Unique product-property records:", f"{len(properties):,}")
display(properties.head())

Unique product-property records: 12,778,737


,itemid,property,value
0,460429,categoryid,1338
1,206783,888,1116713 960601 n277.200
2,395014,400,n552.000 639502 n720.000 424566
3,59481,790,n15360.000
4,156781,917,828513


In [4]:
# Normalize generic property values.

properties["property"] = (
    properties["property"]
    .astype("string")
    .str.strip()
)

properties["value"] = (
    properties["value"]
    .astype("string")
    .str.strip()
)

properties = properties[
    properties["itemid"].notna()
    & properties["property"].notna()
    & properties["value"].notna()
]

print("Clean product-property records:", f"{len(properties):,}")

Clean product-property records: 12,778,737


In [5]:
# Keep metadata flexible instead of assuming unavailable attributes.

metadata_summary = (
    properties
    .groupby("itemid")
    .agg(
        property_count=("property", "nunique"),
        metadata_value_count=("value", "nunique"),
    )
    .reset_index()
    .rename(columns={"itemid": "item_id"})
)

products = products.merge(
    metadata_summary,
    on="item_id",
    how="left"
)

products["property_count"] = (
    products["property_count"]
    .fillna(0)
    .astype(int)
)

products["metadata_value_count"] = (
    products["metadata_value_count"]
    .fillna(0)
    .astype(int)
)

display(products.head())

,item_id,has_interactions,has_metadata,property_count,metadata_value_count
0,0,False,True,28,24
1,1,False,True,35,28
2,2,False,True,24,23
3,3,True,True,29,18
4,4,True,True,25,22


In [6]:
# Product-level behavioral statistics.

product_stats = []

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=["itemid", "event"],
    chunksize=250_000
):
    stats = chunk.groupby("itemid").agg(
        interaction_count=("event", "size"),
        unique_event_types=("event", "nunique"),
    )

    product_stats.append(stats)

product_stats = (
    pd.concat(product_stats)
    .groupby("itemid")
    .sum()
    .reset_index()
    .rename(columns={"itemid": "item_id"})
)

products = products.merge(
    product_stats,
    on="item_id",
    how="left"
)

products["interaction_count"] = (
    products["interaction_count"]
    .fillna(0)
    .astype(int)
)

products["unique_event_types"] = (
    products["unique_event_types"]
    .fillna(0)
    .astype(int)
)

print("Product statistics added.")
display(products.head())

Product statistics added.


,item_id,has_interactions,has_metadata,property_count,metadata_value_count,interaction_count,unique_event_types
0,0,False,True,28,24,0,0
1,1,False,True,35,28,0,0
2,2,False,True,24,23,0,0
3,3,True,True,29,18,2,2
4,4,True,True,25,22,3,3


In [7]:
# Save canonical product dataset.

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

products.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Products:", f"{len(products):,}")

Saved: f:\annuspeaks.com\recommendation-system\data\processed\product_dataset.csv
Products: 466,868


In [8]:
# Final Phase 2.3 validation.

required_columns = [
    "item_id",
    "has_interactions",
    "has_metadata",
    "property_count",
    "metadata_value_count",
    "interaction_count",
    "unique_event_types",
]

missing = [
    column for column in required_columns
    if column not in products.columns
]

print("Missing required columns:", missing)
print("Products:", f"{products['item_id'].nunique():,}")
print("Products with interactions:", f"{products['has_interactions'].sum():,}")
print("Products with metadata:", f"{products['has_metadata'].sum():,}")

assert not missing
assert products["item_id"].notna().all()
assert products["item_id"].is_unique

Missing required columns: []
Products: 466,868
Products with interactions: 235,061
Products with metadata: 417,053
